In [ ]:
import os

SEED = 42

os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import random
import numpy as np
import tensorflow as tf

random.seed(SEED)
np.random.seed(SEED)

tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

print("Deterministic seed =", SEED)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if 'Raw measurements' in f:
            print("✅", os.path.join(root, f))

In [ ]:
import pandas as pd

PATH = "/content/drive/MyDrive/Raw measurements. csv"
df = pd.read_csv(PATH)

print("✅ Dataset loaded")
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

print("\n📌 Column names:")
print(df.columns.tolist())

print("\n📌 First 5 rows:")
display(df.head())

print("\n📌 Missing values:")
print(df.isnull().sum())

In [ ]:
# ==============================
# STEP 1: Create Timestamp
# ==============================

import pandas as pd
import numpy as np

df['Timestamp'] = pd.to_datetime(
    df['Date'].astype(str) + ' ' + df['Time'].astype(str),
    errors='coerce'
)

# Drop rows without a valid timestamp
df = df.dropna(subset=['Timestamp']).copy()

# Sort each device chronologically within each room
df = df.sort_values(
    ['roomcode', 'devcode', 'Timestamp']
).reset_index(drop=True)

print("✅ Timestamp created")
print("Rows:", len(df))
print("Rooms:", df['roomcode'].nunique())
print("Devices:", df['devcode'].nunique())

print("\nFirst rows:")
print(df[['roomcode', 'devcode', 'Timestamp']].head(10))

In [ ]:
# ==========================================
# STEP 2: Check sampling intervals per device
# ==========================================

temp = df[['roomcode', 'devcode', 'Timestamp']].copy()

temp['gap_min'] = (
    temp.groupby(['roomcode', 'devcode'])['Timestamp']
        .diff()
        .dt.total_seconds() / 60
)

# Exclude implausible values and long gaps from the distribution plot
gaps = temp.loc[
    (temp['gap_min'] > 0) &
    (temp['gap_min'] <= 60),
    'gap_min'
]

print("Median interval:", round(gaps.median(), 2), "minutes")

print("\nMost common intervals:")
print(
    gaps.round(2)
        .value_counts()
        .head(20)
)

In [ ]:
# ============================================================
# STEP 3 FINAL: CONTINUOUS 3-MIN RESAMPLING
# No device mixing + No interpolation across gaps > 15 minutes
# ============================================================

import pandas as pd
import numpy as np

SENSOR_COLS = ['CO2', 'PM2.5', 'PM10', 'T', 'RH']
MAX_GAP_MIN = 15
RESAMPLE_RULE = '3min'

# Work from the original data
work = df[
    ['roomcode', 'devcode', 'Timestamp'] + SENSOR_COLS
].copy()

work = work.dropna(subset=['roomcode', 'devcode', 'Timestamp'])
work = work.sort_values(
    ['roomcode', 'devcode', 'Timestamp']
).reset_index(drop=True)

# ------------------------------------------------------------
# 1. Identify interruptions within each device
# ------------------------------------------------------------
gap = (
    work.groupby(['roomcode', 'devcode'])['Timestamp']
        .diff()
        .dt.total_seconds()
        .div(60)
)

new_segment = (
    gap.isna() |
    (gap > MAX_GAP_MIN)
)

work['segment_id'] = (
    new_segment.groupby(
        [work['roomcode'], work['devcode']]
    ).cumsum()
)

print("Original rows :", len(work))
print("Segments      :", work[
    ['roomcode','devcode','segment_id']
].drop_duplicates().shape[0])

# ------------------------------------------------------------
# 2. Resampling within each continuous segment only
# ------------------------------------------------------------
parts = []

for (room, dev, seg), g in work.groupby(
    ['roomcode', 'devcode', 'segment_id'],
    sort=False
):

    if len(g) < 2:
        continue

    g = (
        g.set_index('Timestamp')[SENSOR_COLS]
        .resample(RESAMPLE_RULE)
        .mean()
    )

    # Short gaps only: at most 4 missing points
    MAX_MISSING_BINS = 4

    for col in SENSOR_COLS:

        s = g[col].copy()

        interp = s.interpolate(
            method='linear',
            limit_area='inside'
        )

        is_nan = s.isna()

        run_id = (is_nan != is_nan.shift()).cumsum()

        run_sizes = (
            is_nan.groupby(run_id)
            .transform('sum')
        )

        short_gap = is_nan & (run_sizes <= MAX_MISSING_BINS)

        s.loc[short_gap] = interp.loc[short_gap]

        g[col] = s

    # Drop rows that still have a missing sensor value
    g = g.dropna(subset=SENSOR_COLS)

    if len(g) == 0:
        continue

    # Restore Timestamp from the index
    g = g.rename_axis('Timestamp').reset_index()

    g['roomcode'] = room
    g['devcode'] = dev
    g['segment_id'] = seg

    parts.append(g)

# ------------------------------------------------------------
# 3. Final resampled dataset
# ------------------------------------------------------------
df_resampled = pd.concat(parts, ignore_index=True)

df_resampled = df_resampled[
    ['roomcode', 'devcode', 'segment_id',
     'Timestamp'] + SENSOR_COLS
]

df_resampled = df_resampled.sort_values(
    ['roomcode', 'devcode', 'segment_id', 'Timestamp']
).reset_index(drop=True)
# Re-segment after removing long missing runs
df_resampled = df_resampled.sort_values(
    ['roomcode', 'devcode', 'segment_id', 'Timestamp']
).reset_index(drop=True)

gap_after = (
    df_resampled
    .groupby(['roomcode', 'devcode'])['Timestamp']
    .diff()
    .dt.total_seconds()
    .div(60)
)

new_break = gap_after.isna() | (gap_after > 15)

df_resampled['segment_id'] = (
    new_break
    .groupby([
        df_resampled['roomcode'],
        df_resampled['devcode']
    ])
    .cumsum()
    .astype('int32')
)
# ------------------------------------------------------------
# 4. Verification
# ------------------------------------------------------------
check_gap = (
    df_resampled
    .groupby(['roomcode','devcode','segment_id'])['Timestamp']
    .diff()
    .dt.total_seconds()
    .div(60)
    .dropna()
)

print("\n✅ Corrected resampling completed")
print("Resampled rows :", len(df_resampled))
print("Rooms          :", df_resampled['roomcode'].nunique())
print("Devices        :", df_resampled['devcode'].nunique())

print("\nIntervals inside segments:")
print(check_gap.value_counts().head())

print("\nMissing values:")
print(df_resampled[SENSOR_COLS].isna().sum())

In [ ]:
SENSOR_COLS = ['CO2', 'PM2.5', 'PM10', 'T', 'RH']

# 1) Room 302b and device 26: what happened to them
r = work[(work['roomcode'] == '302b') & (work['devcode'] == 26)]
print("Raw rows (302b, dev 26):", len(r))
print("Missing share per sensor:\n", r[SENSOR_COLS].isna().mean().round(2))

# 2) Share of interpolated records
w = work.copy()
w['bin'] = w['Timestamp'].dt.floor('3min')
real = (w.groupby(['roomcode', 'devcode', 'bin'])[SENSOR_COLS]
          .count().gt(0).all(axis=1).rename('real').reset_index())
m = df_resampled.merge(real, left_on=['roomcode', 'devcode', 'Timestamp'],
                       right_on=['roomcode', 'devcode', 'bin'], how='left')
m['real'] = m['real'].fillna(False)
print("\nInterpolated share: %.1f%%" % (100 * (1 - m['real'].mean())))

In [ ]:
r = df[df['roomcode'] == '302b']
print("302b raw rows:", len(r), "| devices:", r['devcode'].unique())
print(r[['CO2','PM2.5','PM10','T','RH']].notna().sum())

d = df[df['devcode'] == 26]
print("\nDevice 26 raw rows:", len(d), "| rooms:", d['roomcode'].unique())
print(d[['CO2','PM2.5','PM10','T','RH']].notna().sum())

In [ ]:
# ============================================================
# STEP 4: 8-HOUR PREDICTED MOVING AVERAGES
# Within each room + device + continuous segment only
# ============================================================

import pandas as pd
import numpy as np

data = df_resampled.copy()

GROUP = ['roomcode', 'devcode', 'segment_id']

data = data.sort_values(
    GROUP + ['Timestamp']
).reset_index(drop=True)

parts_step4 = []

for _, g in data.groupby(GROUP, sort=False):

    g = g.sort_values('Timestamp').copy()
    g = g.set_index('Timestamp')

    for col in ['CO2', 'PM2.5', 'PM10']:

        # Past 4-hour average
        avg4 = g[col].rolling(
            '4h',
            min_periods=2
        ).mean()

        # Past 1-hour average
        avg1 = g[col].rolling(
            '1h',
            min_periods=2
        ).mean()

        # IAQI-S predicted 8-hour average
        g[f'{col}_8h'] = (4 * avg4 + 4 * avg1) / 8

    g = g.reset_index()
    parts_step4.append(g)

data = pd.concat(parts_step4, ignore_index=True)

# Drop rows without complete averages
before = len(data)

data = data.dropna(
    subset=['CO2_8h', 'PM2.5_8h', 'PM10_8h']
).reset_index(drop=True)

print("✅ STEP 4 completed")
print("Before :", before)
print("After  :", len(data))

print(
    data[
        ['roomcode', 'devcode', 'segment_id', 'Timestamp',
         'CO2_8h', 'PM2.5_8h', 'PM10_8h']
    ].head()
)

In [ ]:
import numpy as np, pandas as pd
G = ['roomcode','devcode','segment_id']

# ---------- M7 ----------
sz = df_resampled.groupby(G).size()
print("Segments after resampling :", len(sz))
print("Segments with 1 row       :", int((sz == 1).sum()))
print("Rows dropped in Step 4    :", len(df_resampled) - len(data))
print("Segment length (rows) stats:\n", sz.describe().round(1))
rows_full4h = np.maximum(sz - 79, 0).sum()   # 4h = 80 readings at 3 min
print("Rows with a full 4h history: %.1f%%" % (100 * rows_full4h / sz.sum()))

# ---------- M5 (raw data) ----------
for c in ['CO2', 'PM2.5', 'PM10']:
    print(c, "min/max:", df[c].min(), df[c].max())
print("CO2 > 5000      :", int((df['CO2'] > 5000).sum()))
print("CO2 < 400       :", int((df['CO2'] < 400).sum()))
print("CO2 == 400      :", int((df['CO2'] == 400).sum()))
print("PM2.5 >= 254    :", int((df['PM2.5'] >= 254).sum()))
print("PM10  >= 255    :", int((df['PM10'] >= 255).sum()))
print("RH == 0         :", int((df['RH'] == 0).sum()))

# ---------- Which room/device disappeared (67->66 / 48->47) ----------
print("Removed rooms  :", set(df['roomcode']) - set(df_resampled['roomcode']))
print("Removed devices:", set(df['devcode'])  - set(df_resampled['devcode']))

In [ ]:
# ============================================================
# STEP 5 FINAL: IAQI-S Ground Truth
# Continuous linear interpolation — no gaps, no rounding
# ============================================================

import numpy as np
import pandas as pd

N_BEFORE = len(data)
print("Rows before IAQI-S:", N_BEFORE)


# ------------------------------------------------------------
# 1. Continuous IAQI interpolation
# ------------------------------------------------------------

def iaqi_continuous(x, concentration_points):
    """
    Continuous linear interpolation from pollutant concentration
    to IAQI-S index.
    """
    x = np.asarray(x, dtype=np.float32)

    iaqi_points = np.array(
        [0, 50, 100, 200, 300, 400, 500],
        dtype=np.float32
    )

    concentration_points = np.asarray(
        concentration_points,
        dtype=np.float32
    )

    out = np.interp(
        x,
        concentration_points,
        iaqi_points
    )

    return out.astype(np.float32)


# ------------------------------------------------------------
# 2. IAQI-S concentration breakpoints
# ------------------------------------------------------------

PM10_POINTS = [
    0, 20, 75, 150, 220, 300, 380
]

PM25_POINTS = [
    0, 10, 35, 70, 100, 140, 180
]

CO2_POINTS = [
    300, 700, 1000, 2000, 3000, 4000, 5000
]


# ------------------------------------------------------------
# 3. Individual IAQI
# ------------------------------------------------------------

data["IAQI_PM10"] = iaqi_continuous(
    data["PM10_8h"].to_numpy(),
    PM10_POINTS
)

data["IAQI_PM25"] = iaqi_continuous(
    data["PM2.5_8h"].to_numpy(),
    PM25_POINTS
)

data["IAQI_CO2"] = iaqi_continuous(
    data["CO2_8h"].to_numpy(),
    CO2_POINTS
)


# ------------------------------------------------------------
# 4. Composite IAQI-S
# ------------------------------------------------------------

idx = data[
    ["IAQI_PM10", "IAQI_PM25", "IAQI_CO2"]
].to_numpy(dtype=np.float32)

# Maximum individual IAQI
base = np.max(idx, axis=1)

# Number of pollutants above Caution threshold
n_caution = np.sum(idx > 100, axis=1)

# EP1
ep1 = np.zeros(len(data), dtype=np.float32)
ep1[n_caution == 2] = 50
ep1[n_caution == 3] = 75

# EP2
ep2 = (
    np.sum((idx > 100) & (idx <= 200), axis=1) * 5
    + np.sum((idx > 200) & (idx <= 300), axis=1) * 10
    + np.sum((idx > 300) & (idx <= 400), axis=1) * 20
    + np.sum(idx > 400, axis=1) * 40
)

# Final composite
data["IAQI_Composite"] = np.clip(
    base + ep1 + ep2,
    0,
    500
)


# ------------------------------------------------------------
# 5. Six IAQI-S classes
# ------------------------------------------------------------

labels = np.array([
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
])

score = data["IAQI_Composite"].to_numpy()

# <=50, >50-100, >100-200, >200-300, >300-400, >400
class_id = np.digitize(
    score,
    bins=[50, 100, 200, 300, 400],
    right=True
)

data["IAQI_Class"] = labels[class_id]


# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

N_AFTER = len(data)

print("\n✅ IAQI-S Ground Truth created")
print("Rows before:", N_BEFORE)
print("Rows after :", N_AFTER)

assert N_AFTER == N_BEFORE, "ERROR: number of rows changed!"


print("\nClass distribution:")

counts = data["IAQI_Class"].value_counts()

for cls in labels:
    n = counts.get(cls, 0)
    p = n / N_AFTER * 100

    print(f"{cls:16s}: {n:,} ({p:.2f}%)")

In [ ]:
# EVT analysis: runs declustering, GPD fit with a censored upper bound (500), CIs and threshold stability
import numpy as np, pandas as pd, json
from scipy.optimize import minimize
import matplotlib.pyplot as plt

df_evt = data[['roomcode', 'devcode', 'segment_id', 'IAQI_Composite']].reset_index(drop=True)
x = df_evt['IAQI_Composite'].values
seg = df_evt.groupby(['roomcode', 'devcode', 'segment_id']).ngroup().values
CAP = 500.0
rng = np.random.default_rng(0)

# negative log-likelihood of the GPD; values at the cap are treated as right-censored
def nll(p, y, cens):
    xi, ls = p
    s = np.exp(ls)
    if abs(xi) < 1e-8:
        xi = 1e-8
    z = 1 + xi * y / s
    if np.any(z <= 0):
        return 1e12
    ll_u = -np.log(s) - (1 / xi + 1) * np.log(z[~cens])
    ll_c = -(1 / xi) * np.log(z[cens])
    return -(ll_u.sum() + ll_c.sum())

def fit(y, cens=None):
    if cens is None:
        cens = np.zeros(len(y), bool)
    m = y[~cens].mean() if (~cens).any() else y.mean()
    r = minimize(nll, [0.1, np.log(max(m, 1e-3))], args=(y, cens),
                 method='Nelder-Mead', options={'xatol': 1e-6, 'fatol': 1e-8, 'maxiter': 2000})
    return r.x[0], np.exp(r.x[1])

# runs declustering: a new cluster starts after a gap longer than r steps or in a new segment
def clusters(u, r):
    ex = np.where(x > u)[0]
    br = np.ones(len(ex), bool)
    br[1:] = (np.diff(ex) > r) | (seg[ex][1:] != seg[ex][:-1])
    cid = np.cumsum(br) - 1
    return len(ex), pd.DataFrame({'c': cid, 'v': x[ex]}).groupby('c').v.max().values

# bootstrap over cluster maxima to get 95% confidence intervals
def boot(y, cens, B=300):
    out = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        out.append(fit(y[i], cens[i]))
    return np.percentile(np.array(out), [2.5, 97.5], axis=0)

u = np.percentile(x, 97.5)
ex = x[x > u] - u
xi, s = fit(ex)
print(f"Naive (all {len(ex)} exceedances): xi={xi:.3f} sigma={s:.1f}")

for r in [20, 80, 160, 480]:
    n, m = clusters(u, r)
    y = m - u
    cens = m >= CAP
    xi0, s0 = fit(y)
    xi1, s1 = fit(y, cens)
    c0 = boot(y, np.zeros(len(y), bool))
    c1 = boot(y, cens)
    print(f"r={r}: clusters={len(m)} at_cap={cens.sum()}")
    print(f"  ignoring cap : xi={xi0:.3f} [{c0[0,0]:.3f},{c0[1,0]:.3f}]  sigma={s0:.1f} [{c0[0,1]:.1f},{c0[1,1]:.1f}]")
    print(f"  cap censored : xi={xi1:.3f} [{c1[0,0]:.3f},{c1[1,0]:.3f}]  sigma={s1:.1f} [{c1[0,1]:.1f},{c1[1,1]:.1f}]")

# threshold stability (r = 160 steps = 8 h, cap censored)
rows = []
for q in np.arange(90, 99.6, 0.5):
    uq = np.percentile(x, q)
    n, m = clusters(uq, 160)
    if len(m) < 30:
        continue
    y = m - uq
    c = m >= CAP
    xi, s = fit(y, c)
    ci = boot(y, c, 100)
    rows.append((q, uq, len(m), xi, s - xi * uq, ci[0, 0], ci[1, 0]))
st = pd.DataFrame(rows, columns=['q', 'u', 'clusters', 'xi', 'sigma_mod', 'xi_lo', 'xi_hi'])
print(st.round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].plot(st.u, st.xi, 'o-'); ax[0].fill_between(st.u, st.xi_lo, st.xi_hi, alpha=.25)
ax[1].plot(st.u, st.sigma_mod, 'o-')
for a in ax:
    a.axvline(u, ls='--', c='gray'); a.set_xlabel('Threshold u (IAQI-S)')
ax[0].set_ylabel('Shape xi (95% CI)'); ax[1].set_ylabel('Modified scale')
plt.tight_layout(); plt.savefig('/content/drive/MyDrive/IAQ_Project_Saved/evt_stability.png', dpi=200)

In [ ]:
# ============================================================
# STEP 6 FINAL:
# Fixed + deterministic stratified ROOM-BASED split
# Uses only labels that will actually survive WINDOW=10
# No room overlap
# ============================================================

import numpy as np
import pandas as pd

SEED = 42
WINDOW = 10
N_VAL = 5
N_TEST = 5

LABELS = [
    'Good',
    'Moderate',
    'Caution',
    'Unhealthy',
    'Very Unhealthy',
    'Hazardous'
]

GROUP = ['roomcode', 'devcode', 'segment_id']

# Physical room = room code without the retrofit-period suffix (-a/-b/-c)
data['room_group'] = (
    data['roomcode'].astype(str).str.replace(r'[abc]$', '', regex=True)
)

# ------------------------------------------------------------
# 1. Find labels that are actually eligible after WINDOW=10
# ------------------------------------------------------------
eligible_idx = []

for _, g in data.groupby(GROUP, sort=False):

    g = g.sort_values('Timestamp')

    if len(g) < WINDOW:
        continue

    eligible_idx.extend(
        g.index[WINDOW - 1:].tolist()
    )

eligible = data.loc[
    eligible_idx,
    ['room_group', 'IAQI_Class']
].copy()

all_rooms = sorted(
    data['room_group'].dropna().unique().tolist(),
    key=lambda x: str(x)
)

room_counts = (
    eligible
    .groupby(['room_group', 'IAQI_Class'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=all_rooms, fill_value=0)
    .reindex(columns=LABELS, fill_value=0)
)

# ------------------------------------------------------------
# 2. Scientific check:
# each class must exist in at least 3 different rooms
# if we want Train + Validation + Test all to contain it
# ------------------------------------------------------------
room_support = (room_counts > 0).sum(axis=0)

print("Rooms containing each eligible class:")
print(room_support)

if (room_support < 3).any():
    bad = room_support[room_support < 3]

    raise ValueError(
        "\nCannot create a leakage-free room-based "
        "Train/Validation/Test split containing every class.\n"
        f"Classes with fewer than 3 supporting rooms:\n{bad}"
    )

# ------------------------------------------------------------
# 3. Deterministic room selection
#    Not thousands of random trials
# ------------------------------------------------------------
rng = np.random.default_rng(SEED)

random_order = rng.permutation(all_rooms)

tie_rank = {
    room: i
    for i, room in enumerate(random_order)
}

global_counts = room_counts.sum(axis=0).to_numpy(dtype=float)
global_prop = global_counts / global_counts.sum()

def choose_rooms(candidates, n_pick):

    candidates = list(candidates)

    selected = []
    selected_counts = np.zeros(len(LABELS), dtype=float)

    support = (
        (room_counts.loc[candidates] > 0)
        .sum(axis=0)
        .to_numpy(dtype=float)
    )

    # Start with rarest classes
    class_order = np.argsort(support)

    for j in class_order:

        if selected_counts[j] > 0:
            continue

        available = [
            r for r in candidates
            if r not in selected
            and room_counts.loc[r, LABELS[j]] > 0
        ]

        if not available:
            raise ValueError(
                f"No available room contains class {LABELS[j]}"
            )

        def coverage_score(room):

            row = (
                room_counts
                .loc[room, LABELS]
                .to_numpy(dtype=float)
            )

            present = row > 0

            return np.sum(
                present / (support + 1e-9)
            )

        best_room = max(
            available,
            key=lambda r: (
                coverage_score(r),
                -tie_rank[r]
            )
        )

        selected.append(best_room)

        selected_counts += (
            room_counts
            .loc[best_room, LABELS]
            .to_numpy(dtype=float)
        )

        if len(selected) == n_pick:
            return np.array(selected, dtype=object)

    # Fill remaining rooms while keeping distribution close
    while len(selected) < n_pick:

        best_room = None
        best_score = np.inf

        for room in candidates:

            if room in selected:
                continue
            new_counts = (
                selected_counts
                + room_counts
                  .loc[room, LABELS]
                  .to_numpy(dtype=float)
            )

            if new_counts.sum() == 0:
                score = np.inf
            else:
                new_prop = new_counts / new_counts.sum()

                score = np.mean(
                    np.abs(new_prop - global_prop)
                    / np.sqrt(global_prop + 1e-8)
                )

            if (
                score < best_score
                or (
                    np.isclose(score, best_score)
                    and (
                        best_room is None
                        or tie_rank[room] < tie_rank[best_room]
                    )
                )
            ):
                best_score = score
                best_room = room

        selected.append(best_room)

        selected_counts += (
            room_counts
            .loc[best_room, LABELS]
            .to_numpy(dtype=float)
        )

    return np.array(selected, dtype=object)

# Test first
test_rooms = choose_rooms(
    all_rooms,
    N_TEST
)

remaining = [
    r for r in all_rooms
    if r not in set(test_rooms)
]

# Validation from remaining rooms
val_rooms = choose_rooms(
    remaining,
    N_VAL
)

train_rooms = np.array(
    [
        r for r in remaining
        if r not in set(val_rooms)
    ],
    dtype=object
)

# ------------------------------------------------------------
# 4. Create final data partitions
# ------------------------------------------------------------
train_part = data[
    data['room_group'].isin(train_rooms)
].copy()

val_part = data[
    data['room_group'].isin(val_rooms)
].copy()

test_part = data[
    data['room_group'].isin(test_rooms)
].copy()

# ------------------------------------------------------------
# 5. Final verification using WINDOW-eligible labels
# ------------------------------------------------------------
print("\n✅ FINAL FIXED PHYSICAL-ROOM SPLIT")

print("Train rooms      :", len(train_rooms))
print("Validation rooms :", len(val_rooms))
print("Test rooms       :", len(test_rooms))

print("\nRows:")
print("Train      :", len(train_part))
print("Validation :", len(val_part))
print("Test       :", len(test_part))

print("\nRoom overlap:")
print(
    "Train-Val  :",
    len(set(train_rooms) & set(val_rooms))
)
print(
    "Train-Test :",
    len(set(train_rooms) & set(test_rooms))
)
print(
    "Val-Test   :",
    len(set(val_rooms) & set(test_rooms))
)

for name, rooms_part in [
    ("TRAIN", train_rooms),
    ("VAL", val_rooms),
    ("TEST", test_rooms)
]:

    counts = (
        room_counts
        .loc[list(rooms_part), LABELS]
        .sum(axis=0)
    )

    print(f"\n{name} eligible labels after WINDOW={WINDOW}:")

    for cls in LABELS:
        print(f"{cls:16s}: {int(counts[cls]):,}")

    assert (counts > 0).all(), (
        f"{name} is missing one or more classes"
    )

print("\n✅ All 6 classes exist in Train, Validation and Test")

In [ ]:
from sklearn.preprocessing import StandardScaler

features = ['CO2', 'PM2.5', 'PM10', 'T', 'RH']

scaler = StandardScaler()

# Fit ONLY on training data
scaler.fit(train_part[features])

# Transform Train / Validation / Test
train_part[features] = scaler.transform(train_part[features])
val_part[features]   = scaler.transform(val_part[features])
test_part[features]  = scaler.transform(test_part[features])

print("✅ Scaling completed")
print("Features:", features)

print("\nTrain mean after scaling:")
print(train_part[features].mean().round(3))

print("\nTrain std after scaling:")
print(train_part[features].std().round(3))

In [ ]:
# ============================================================
# STEP 8 (corrected): Create 10-reading windows
# Same room + same device + same segment ONLY
# Uses train_part / val_part / test_part directly (from STEP 6)
# ============================================================

import numpy as np

WINDOW =10

class_to_id = {
    'Good': 0,
    'Moderate': 1,
    'Caution': 2,
    'Unhealthy': 3,
    'Very Unhealthy': 4,
    'Hazardous': 5
}

def make_windows(df):

    X_all = []
    y_all = []

    # Every sequence must stay within the same room, device and segment
    for _, g in df.groupby(
        ['roomcode', 'devcode', 'segment_id'],
        sort=False
    ):

        g = g.sort_values('Timestamp')

        if len(g) < WINDOW:
            continue

        X = g[features].to_numpy(dtype=np.float32)
        y = g['IAQI_Class'].map(class_to_id).to_numpy()

        windows = np.lib.stride_tricks.sliding_window_view(
            X, WINDOW, axis=0
        ).transpose(0, 2, 1)

        X_all.append(windows)
        y_all.append(y[WINDOW - 1:])   # current-state label

    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)

    return X_all, y_all


# ⚠️ The correction is here: use train_part / val_part / test_part
# (the same variables produced by the corrected STEP 6), not train_df/val_df/test_df

X_train, y_train = make_windows(train_part)
X_val, y_val     = make_windows(val_part)
X_test, y_test   = make_windows(test_part)

print("✅ 15-reading windows created correctly")

print("\nTrain:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nEach window shape:", X_train.shape[1:])

# ------------------------------------------------------------
# Immediate check: actual class distribution after building the windows
# ------------------------------------------------------------
class_names = ['Good','Moderate','Caution','Unhealthy','Very Unhealthy','Hazardous']

print("\n📊 Actual class distribution after windowing:")
for name, y_set in [("TRAIN", y_train), ("VAL", y_val), ("TEST", y_test)]:
    print(f"\n{name}:")
    for i, cname in enumerate(class_names):
        n = np.sum(y_set == i)
        print(f"  {cname:16s}: {n:,}")

In [ ]:
# ============================================================
# STEP 8b (new): Oversampling of the very rare classes — TRAIN only
# Very Unhealthy and Hazardous only, without touching Val/Test
# ============================================================

import numpy as np

RARE_CLASSES = [4, 5]   # 4=Very Unhealthy, 5=Hazardous
TARGET_COUNT = 5000

rng = np.random.default_rng(42)

X_train_list = [X_train]
y_train_list = [y_train]

for cls in RARE_CLASSES:
    idx = np.where(y_train == cls)[0]
    n_have = len(idx)

    if n_have == 0:
        print(f"⚠️ Warning: class {cls} does not exist in training at all!")
        continue

    n_needed = TARGET_COUNT - n_have
    if n_needed <= 0:
        continue

    chosen = rng.choice(idx, size=n_needed, replace=True)
    X_extra = X_train[chosen].copy()

    noise = rng.normal(0, 0.01, X_extra.shape).astype(np.float32)
    X_extra = X_extra + noise

    y_extra = np.full(n_needed, cls, dtype=y_train.dtype)

    X_train_list.append(X_extra)
    y_train_list.append(y_extra)

    print(f"✅ Class {cls}: was {n_have} → now {n_have + n_needed}")

X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

shuffle_idx = rng.permutation(len(X_train))
X_train = X_train[shuffle_idx]
y_train = y_train[shuffle_idx]

print("\n📊 Final Train distribution after oversampling:")
for i in range(6):
    print(f"  {i}: {np.sum(y_train == i):,}")

print("\n🔒 Check: Val and Test are unchanged")
print("Test Hazardous:", np.sum(y_test == 5))

In [ ]:
import os, time
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

SEEDS = [42, 7, 123, 2024, 1]
SAVE_DIR = "/content/drive/MyDrive/IAQ_Project_Saved/tree_baselines"
os.makedirs(SAVE_DIR, exist_ok=True)
CLS = ["Good", "Moderate", "Caution", "Unhealthy", "VeryUnhealthy", "Hazardous"]

# summarise each 10-step window with last value, mean, max and slope per variable
def window_features(X):
    t = np.arange(X.shape[1], dtype=np.float32)
    t -= t.mean()
    last = X[:, -1, :]
    mean = X.mean(axis=1)
    mx = X.max(axis=1)
    slope = (X * t[None, :, None]).sum(axis=1) / (t ** 2).sum()
    return np.hstack([last, mean, mx, slope]).astype(np.float32)

F_train, F_val, F_test = window_features(X_train), window_features(X_val), window_features(X_test)

csv_path = f"{SAVE_DIR}/results_trees.csv"
done = set()
if os.path.exists(csv_path):
    d0 = pd.read_csv(csv_path)
    done = set(zip(d0["model"], d0["seed"]))

for seed in SEEDS:
    for name in ["RandomForest", "LightGBM"]:
        if (name, seed) in done:
            print(f"{name} seed {seed}: already done")
            continue
        t0 = time.time()
        if name == "LightGBM":
            clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, num_leaves=63,
                                     subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                     random_state=seed, n_jobs=-1, verbose=-1)
        else:
            clf = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, max_samples=0.2,
                                         n_jobs=-1, random_state=seed)
        clf.fit(F_train, y_train)
        pv, pt = clf.predict(F_val), clf.predict(F_test)
        row = {"model": name, "seed": seed,
               "val_macro_f1": f1_score(y_val, pv, average="macro"),
               "test_acc": accuracy_score(y_test, pt) * 100,
               "test_macro_f1": f1_score(y_test, pt, average="macro"),
               "minutes": (time.time() - t0) / 60}
        f1c = f1_score(y_test, pt, average=None, labels=np.arange(6), zero_division=0)
        for k, n in enumerate(CLS):
            row["f1_" + n] = float(f1c[k])
        pd.DataFrame([row]).to_csv(csv_path, mode="a", header=not os.path.exists(csv_path), index=False)
        print(f"{name:13s} seed {seed:5d} | test macro-F1 {row['test_macro_f1']:.4f} | "
              f"acc {row['test_acc']:.2f} | {row['minutes']:.1f} min")

res = pd.read_csv(csv_path)
print(res.groupby("model")[["test_macro_f1", "test_acc"]].agg(["mean", "std"]).round(4))

In [ ]:
import os, time, json
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers as L
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import f1_score, accuracy_score

SEEDS = [7, 123, 2024, 1]
SAVE_DIR = "/content/drive/MyDrive/IAQ_Project_Saved/multi_seed_all"
os.makedirs(SAVE_DIR, exist_ok=True)
CLS = ["Good","Moderate","Caution","Unhealthy","VeryUnhealthy","Hazardous"]

def tcn_block(x, filters=64, kernel_size=3, dilation_rate=2, dropout=0.2):
    res = x
    x = L.Conv1D(filters, kernel_size, padding="causal", dilation_rate=dilation_rate)(x)
    x = L.BatchNormalization()(x); x = L.Activation("relu")(x); x = L.Dropout(dropout)(x)
    x = L.Conv1D(filters, kernel_size, padding="causal", dilation_rate=dilation_rate)(x)
    x = L.BatchNormalization()(x)
    if res.shape[-1] != filters:
        res = L.Conv1D(filters, 1, padding="same")(res)
    return L.Activation("relu")(L.Add()([x, res]))

def head(x):
    x = L.Dropout(0.2)(L.Dense(64, activation="relu")(x))
    return L.Dense(6, activation="softmax")(x)

def cbr(x):
    return L.Dropout(0.2)(L.Activation("relu")(L.BatchNormalization()(L.Conv1D(64, 3, padding="same")(x))))

def b_lstm():
    i = L.Input(shape=X_train.shape[1:])
    x = L.Dropout(0.2)(L.LSTM(64, return_sequences=True)(i))
    x = L.Dropout(0.2)(L.LSTM(32)(x))
    return Model(i, head(x))

def b_cnn():
    i = L.Input(shape=X_train.shape[1:])
    x = L.Activation("relu")(L.BatchNormalization()(L.Conv1D(64, 3, padding="same")(i)))
    x = L.Activation("relu")(L.BatchNormalization()(L.Conv1D(64, 3, padding="same")(x)))
    x = L.Dropout(0.2)(L.MaxPooling1D(2)(x))
    x = L.Activation("relu")(L.BatchNormalization()(L.Conv1D(128, 3, padding="same")(x)))
    return Model(i, head(L.GlobalAveragePooling1D()(x)))

def b_tcn():
    i = L.Input(shape=X_train.shape[1:])
    x = i
    for d in (1, 2, 4):
        x = tcn_block(x, dilation_rate=d)
    return Model(i, head(L.GlobalAveragePooling1D()(x)))

def b_cnn_lstm():
    i = L.Input(shape=X_train.shape[1:])
    x = L.Dropout(0.2)(L.LSTM(64)(cbr(i)))
    return Model(i, head(x))

def b_cnn_tcn():
    i = L.Input(shape=X_train.shape[1:])
    x = tcn_block(cbr(i))
    return Model(i, head(L.GlobalAveragePooling1D()(x)))

def b_lstm_tcn():
    i = L.Input(shape=X_train.shape[1:])
    x = L.Dropout(0.2)(L.LSTM(64, return_sequences=True)(i))
    x = tcn_block(x)
    return Model(i, head(L.GlobalAveragePooling1D()(x)))

def b_parallel():
    i = L.Input(shape=X_train.shape[1:])
    c = L.GlobalAveragePooling1D()(cbr(i))
    l = L.Dropout(0.2)(L.LSTM(64)(i))
    t = L.GlobalAveragePooling1D()(tcn_block(i))
    return Model(i, head(L.Concatenate()([c, l, t])))

def b_mobile():
    i = L.Input(shape=X_train.shape[1:])
    x = L.BatchNormalization()(L.Conv1D(64, 3, padding="same", activation="relu")(i))
    for f in (64, 128, 128):
        r = x
        x = L.BatchNormalization()(L.SeparableConv1D(f, 3, padding="same", activation="relu")(x))
        x = L.BatchNormalization()(L.SeparableConv1D(f, 3, padding="same", activation="relu")(x))
        if r.shape[-1] != f:
            r = L.Conv1D(f, 1, padding="same")(r)
        x = L.Add()([x, r])
    return Model(i, head(L.GlobalAveragePooling1D()(x)))

MODELS = {"LSTM": b_lstm, "CNN1D": b_cnn, "TCN": b_tcn, "CNN_LSTM": b_cnn_lstm,
          "CNN_TCN": b_cnn_tcn, "LSTM_TCN": b_lstm_tcn, "Parallel": b_parallel,
          "MobileNet1D": b_mobile}

csv_path = f"{SAVE_DIR}/results_all.csv"
done = set()
if os.path.exists(csv_path):
    d0 = pd.read_csv(csv_path); done = set(zip(d0["model"], d0["seed"]))

for seed in SEEDS:
    for name, build in MODELS.items():
        if (name, seed) in done:
            print(f"{name} seed {seed}: already done"); continue
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)
        m = build()
        m.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        cb = [EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True),
              ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6)]
        t0 = time.time()
        h = m.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15,
                  batch_size=256, shuffle=True, callbacks=cb, verbose=0)
        minutes = (time.time() - t0) / 60
        pv = m.predict(X_val, batch_size=1024, verbose=0)
        pt = m.predict(X_test, batch_size=1024, verbose=0)
        yv, yt = pv.argmax(1), pt.argmax(1)
        row = {"model": name, "seed": seed,
               "val_macro_f1": f1_score(y_val, yv, average="macro"),
               "test_acc": accuracy_score(y_test, yt) * 100,
               "test_macro_f1": f1_score(y_test, yt, average="macro"),
               "epochs_run": len(h.history["loss"]), "minutes": minutes}
        f1c = f1_score(y_test, yt, average=None, labels=np.arange(6), zero_division=0)
        for k, n in enumerate(CLS):
            row["f1_" + n] = float(f1c[k])
        pd.DataFrame([row]).to_csv(csv_path, mode="a", header=not os.path.exists(csv_path), index=False)
        np.save(f"{SAVE_DIR}/test_prob_{name}_seed{seed}.npy", pt.astype(np.float16))
        print(f"{name:12s} seed {seed:5d} | test macro-F1 {row['test_macro_f1']:.4f} | "
              f"acc {row['test_acc']:.2f} | epochs {row['epochs_run']} | {minutes:.1f} min")

res = pd.read_csv(csv_path)
print(res.groupby("model")[["test_macro_f1", "test_acc"]].agg(["mean", "std"]).round(4))

In [ ]:
# ============================================================
# STEP 9 FINAL
# Oversampling is already used -> NO class weights
# ============================================================

class_weights = None

print("✅ Oversampling only")
print("✅ Class weights disabled")

In [ ]:
# ============================================================
# MULTI-SEED: CNN + LSTM + TCN
# ============================================================
import os, time, json
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, Activation,
                                     LSTM, Dropout, Add, GlobalAveragePooling1D, Dense)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import f1_score, accuracy_score

SEEDS = [42, 7, 123, 2024, 1]
SAVE_DIR = "/content/drive/MyDrive/IAQ_Project_Saved/multi_seed_triple"
os.makedirs(SAVE_DIR, exist_ok=True)

def tcn_block(x, filters=64, kernel_size=3, dilation_rate=2, dropout=0.2):
    res = x
    x = Conv1D(filters, kernel_size, padding="causal", dilation_rate=dilation_rate)(x)
    x = BatchNormalization()(x); x = Activation("relu")(x); x = Dropout(dropout)(x)
    x = Conv1D(filters, kernel_size, padding="causal", dilation_rate=dilation_rate)(x)
    x = BatchNormalization()(x)
    if res.shape[-1] != filters:
        res = Conv1D(filters, 1, padding="same")(res)
    return Activation("relu")(Add()([x, res]))

def build_triple():
    i = Input(shape=X_train.shape[1:])
    x = Conv1D(64, 3, padding="same")(i)
    x = Dropout(0.2)(Activation("relu")(BatchNormalization()(x)))
    x = Dropout(0.2)(LSTM(64, return_sequences=True)(x))
    x = tcn_block(x)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.2)(Dense(64, activation="relu")(x))
    return Model(i, Dense(6, activation="softmax")(x))

csv_path = f"{SAVE_DIR}/results_triple.csv"
done = set(pd.read_csv(csv_path)["seed"]) if os.path.exists(csv_path) else set()

for seed in SEEDS:
    if seed in done:
        print(f"seed {seed}: already done, skipping"); continue
    print(f"\n===== seed {seed} =====")
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    m = build_triple()
    m.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cb = [EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True),
          ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6)]
    t0 = time.time()
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15,
              batch_size=256, class_weight=None, shuffle=True, callbacks=cb, verbose=2)
    minutes = (time.time() - t0) / 60

    pv = m.predict(X_val,  batch_size=256, verbose=0)
    pt = m.predict(X_test, batch_size=256, verbose=0)
    yv, yt = np.argmax(pv, 1), np.argmax(pt, 1)
    test_loss = float(tf.keras.losses.sparse_categorical_crossentropy(y_test, pt).numpy().mean())
    f1c = f1_score(y_test, yt, average=None, labels=np.arange(6), zero_division=0)

    row = {"seed": seed,
           "val_acc": accuracy_score(y_val, yv) * 100,
           "val_macro_f1": f1_score(y_val, yv, average="macro"),
           "test_acc": accuracy_score(y_test, yt) * 100,
           "test_macro_f1": f1_score(y_test, yt, average="macro"),
           "test_weighted_f1": f1_score(y_test, yt, average="weighted"),
           "test_loss": test_loss,
           "epochs_run": len(h.history["loss"]), "minutes": minutes}
    for k, n in enumerate(["Good","Moderate","Caution","Unhealthy","VeryUnhealthy","Hazardous"]):
        row["f1_" + n] = f1c[k]
    print({k: round(v, 4) for k, v in row.items()})

    pd.DataFrame([row]).to_csv(csv_path, mode="a", header=not os.path.exists(csv_path), index=False)
    json.dump({k: [float(v) for v in vals] for k, vals in h.history.items()},
              open(f"{SAVE_DIR}/history_seed{seed}.json", "w"))
    np.save(f"{SAVE_DIR}/test_prob_seed{seed}.npy", pt.astype(np.float32))

df = pd.read_csv(csv_path)
print("\n===== mean ± SD over", len(df), "seeds =====")
for c in ["val_macro_f1", "test_macro_f1", "test_acc", "test_weighted_f1", "f1_Hazardous", "minutes"]:
    print(f"{c:18s}: {df[c].mean():.4f} ± {df[c].std(ddof=1):.4f}   (min {df[c].min():.4f}, max {df[c].max():.4f})")
print("\nSaved in:", SAVE_DIR)

In [ ]:
# ============================================================
# SINGLE LSTM MODEL
# Build + Train + Evaluate
# ============================================================

import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    classification_report,
    f1_score
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42
tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# 1. BUILD LSTM MODEL
# ------------------------------------------------------------

inputs = Input(
    shape=X_train.shape[1:],
    name="Input"
)

x = LSTM(
    64,
    return_sequences=True,
    name="LSTM_64"
)(inputs)

x = Dropout(0.2)(x)

x = LSTM(
    32,
    return_sequences=False,
    name="LSTM_32"
)(x)

x = Dropout(0.2)(x)

x = Dense(
    64,
    activation="relu",
    name="Dense_64"
)(x)

x = Dropout(0.2)(x)

outputs = Dense(
    6,
    activation="softmax",
    name="Output"
)(x)

lstm_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="LSTM_Single"
)

# ------------------------------------------------------------
# 2. COMPILE
# ------------------------------------------------------------

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()

# ------------------------------------------------------------
# 3. CALLBACKS
# ------------------------------------------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=1,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

# ------------------------------------------------------------
# 4. TRAIN
# ------------------------------------------------------------

start_time = time.time()

history_lstm = lstm_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=15,
    batch_size=256,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    class_weight=None,
    shuffle=True,
    verbose=1
)

training_time = time.time() - start_time

# ------------------------------------------------------------
# 5. TEST
# ------------------------------------------------------------

test_loss, test_accuracy = lstm_model.evaluate(
    X_test,
    y_test,
    batch_size=256,
    verbose=1
)

# ------------------------------------------------------------
# 6. PREDICTIONS
# ------------------------------------------------------------

y_prob = lstm_model.predict(
    X_test,
    batch_size=256,
    verbose=1
)

y_pred = np.argmax(
    y_prob,
    axis=1
)

# ------------------------------------------------------------
# 7. METRICS
# ------------------------------------------------------------

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

print("\n" + "=" * 65)
print("LSTM SINGLE MODEL - FINAL TEST RESULT")
print("=" * 65)

print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Weighted F1   : {weighted_f1:.4f}")
print(f"Training Time : {training_time / 60:.2f} minutes")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# SINGLE 1D-CNN MODEL
# Build + Train + Evaluate
# ============================================================

import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    MaxPooling1D,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    classification_report,
    f1_score
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42
tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# 1. BUILD 1D-CNN
# ------------------------------------------------------------

inputs = Input(
    shape=X_train.shape[1:],
    name="Input"
)

x = Conv1D(
    64,
    kernel_size=3,
    padding="same",
    name="Conv1D_64_1"
)(inputs)

x = BatchNormalization()(x)
x = Activation("relu")(x)

x = Conv1D(
    64,
    kernel_size=3,
    padding="same",
    name="Conv1D_64_2"
)(x)

x = BatchNormalization()(x)
x = Activation("relu")(x)

x = MaxPooling1D(
    pool_size=2,
    name="MaxPooling"
)(x)

x = Dropout(0.2)(x)

x = Conv1D(
    128,
    kernel_size=3,
    padding="same",
    name="Conv1D_128"
)(x)

x = BatchNormalization()(x)
x = Activation("relu")(x)

x = GlobalAveragePooling1D()(x)

x = Dense(
    64,
    activation="relu",
    name="Dense_64"
)(x)

x = Dropout(0.2)(x)

outputs = Dense(
    6,
    activation="softmax",
    name="Output"
)(x)

cnn_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="CNN_1D_Single"
)

# ------------------------------------------------------------
# 2. COMPILE
# ------------------------------------------------------------

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

# ------------------------------------------------------------
# 3. CALLBACKS
# ------------------------------------------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=1,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

# ------------------------------------------------------------
# 4. TRAIN
# ------------------------------------------------------------

start_time = time.time()

history_cnn = cnn_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=15,
    batch_size=256,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    class_weight=None,
    shuffle=True,
    verbose=1
)

training_time = time.time() - start_time

# ------------------------------------------------------------
# 5. TEST
# ------------------------------------------------------------

test_loss, test_accuracy = cnn_model.evaluate(
    X_test,
    y_test,
    batch_size=256,
    verbose=1
)

# ------------------------------------------------------------
# 6. PREDICTIONS
# ------------------------------------------------------------

y_prob = cnn_model.predict(
    X_test,
    batch_size=256,
    verbose=1
)

y_pred = np.argmax(
    y_prob,
    axis=1
)

# ------------------------------------------------------------
# 7. METRICS
# ------------------------------------------------------------

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

print("\n" + "=" * 65)
print("1D-CNN SINGLE MODEL - FINAL TEST RESULT")
print("=" * 65)
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Weighted F1   : {weighted_f1:.4f}")
print(f"Training Time : {training_time / 60:.2f} minutes")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# SINGLE TCN MODEL
# Build + Train + Evaluate
# ============================================================

import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    Dropout,
    Add,
    GlobalAveragePooling1D,
    Dense
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    classification_report,
    f1_score
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42
tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# TCN Residual Block
# ------------------------------------------------------------

def tcn_block(
    x,
    filters=64,
    kernel_size=3,
    dilation_rate=1,
    dropout=0.2
):

    residual = x

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate
    )(x)

    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(dropout)(x)

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate
    )(x)

    x = BatchNormalization()(x)

    # Match residual channels if needed
    if residual.shape[-1] != filters:
        residual = Conv1D(
            filters,
            kernel_size=1,
            padding="same"
        )(residual)

    x = Add()([x, residual])
    x = Activation("relu")(x)

    return x


# ------------------------------------------------------------
# 1. BUILD TCN
# ------------------------------------------------------------

inputs = Input(
    shape=X_train.shape[1:],
    name="Input"
)

x = tcn_block(
    inputs,
    filters=64,
    kernel_size=3,
    dilation_rate=1,
    dropout=0.2
)

x = tcn_block(
    x,
    filters=64,
    kernel_size=3,
    dilation_rate=2,
    dropout=0.2
)

x = tcn_block(
    x,
    filters=64,
    kernel_size=3,
    dilation_rate=4,
    dropout=0.2
)

x = GlobalAveragePooling1D()(x)

x = Dense(
    64,
    activation="relu",
    name="Dense_64"
)(x)

x = Dropout(0.2)(x)

outputs = Dense(
    6,
    activation="softmax",
    name="Output"
)(x)

tcn_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="TCN_Single"
)

# ------------------------------------------------------------
# 2. COMPILE
# ------------------------------------------------------------

tcn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

tcn_model.summary()

# ------------------------------------------------------------
# 3. CALLBACKS
# ------------------------------------------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=1,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

# ------------------------------------------------------------
# 4. TRAIN
# ------------------------------------------------------------

start_time = time.time()

history_tcn = tcn_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=15,
    batch_size=256,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    class_weight=None,
    shuffle=True,
    verbose=1
)

training_time = time.time() - start_time

# ------------------------------------------------------------
# 5. TEST
# ------------------------------------------------------------

test_loss, test_accuracy = tcn_model.evaluate(
    X_test,
    y_test,
    batch_size=256,
    verbose=1
)

# ------------------------------------------------------------
# 6. PREDICTIONS
# ------------------------------------------------------------

y_prob = tcn_model.predict(
    X_test,
    batch_size=256,
    verbose=1
)

y_pred = np.argmax(
    y_prob,
    axis=1
)

# ------------------------------------------------------------
# 7. METRICS
# ------------------------------------------------------------

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

print("\n" + "=" * 65)
print("TCN SINGLE MODEL - FINAL TEST RESULT")
print("=" * 65)

print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Weighted F1   : {weighted_f1:.4f}")
print(f"Training Time : {training_time / 60:.2f} minutes")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# CNN + LSTM HYBRID
# 1 CNN + 1 LSTM
# Build + Train + Evaluate
# ============================================================

import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    classification_report,
    f1_score
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42
tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# 1. BUILD MODEL
# Input -> CNN -> LSTM -> Dense -> Output
# ------------------------------------------------------------

inputs = Input(
    shape=X_train.shape[1:],
    name="Input"
)

# CNN
x = Conv1D(
    filters=64,
    kernel_size=3,
    padding="same",
    name="CNN_64"
)(inputs)

x = BatchNormalization()(x)
x = Activation("relu")(x)
x = Dropout(0.2)(x)

# LSTM
x = LSTM(
    64,
    return_sequences=False,
    name="LSTM_64"
)(x)

x = Dropout(0.2)(x)

# Dense
x = Dense(
    64,
    activation="relu",
    name="Dense_64"
)(x)

x = Dropout(0.2)(x)

# Output
outputs = Dense(
    6,
    activation="softmax",
    name="Output"
)(x)

cnn_lstm_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="CNN_LSTM_1x1"
)

# ------------------------------------------------------------
# 2. COMPILE
# ------------------------------------------------------------

cnn_lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_lstm_model.summary()

# ------------------------------------------------------------
# 3. CALLBACKS
# ------------------------------------------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=1,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

# ------------------------------------------------------------
# 4. TRAIN
# ------------------------------------------------------------

start_time = time.time()

history_cnn_lstm = cnn_lstm_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=15,
    batch_size=256,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    class_weight=None,
    shuffle=True,
    verbose=1
)

training_time = time.time() - start_time

# ------------------------------------------------------------
# 5. TEST
# ------------------------------------------------------------

test_loss, test_accuracy = cnn_lstm_model.evaluate(
    X_test,
    y_test,
    batch_size=256,
    verbose=1
)

# ------------------------------------------------------------
# 6. PREDICTIONS
# ------------------------------------------------------------

y_prob = cnn_lstm_model.predict(
    X_test,
    batch_size=256,
    verbose=1
)

y_pred = np.argmax(
    y_prob,
    axis=1
)

# ------------------------------------------------------------
# 7. METRICS
# ------------------------------------------------------------

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

print("\n" + "=" * 65)
print("CNN + LSTM HYBRID - FINAL TEST RESULT")
print("=" * 65)

print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Weighted F1   : {weighted_f1:.4f}")
print(f"Training Time : {training_time / 60:.2f} minutes")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# CNN + TCN HYBRID
# 1 CNN + 1 TCN Block
# Build + Train + Evaluate
# ============================================================

import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    Dropout,
    Add,
    GlobalAveragePooling1D,
    Dense
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    classification_report,
    f1_score
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42
tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# TCN BLOCK
# ------------------------------------------------------------

def tcn_block(x, filters=64, kernel_size=3, dilation_rate=2, dropout=0.2):

    residual = x

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        name="TCN_Conv1"
    )(x)

    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(dropout)(x)

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        name="TCN_Conv2"
    )(x)

    x = BatchNormalization()(x)

    if residual.shape[-1] != filters:
        residual = Conv1D(
            filters,
            kernel_size=1,
            padding="same"
        )(residual)

    x = Add()([x, residual])
    x = Activation("relu")(x)

    return x


# ------------------------------------------------------------
# 1. BUILD MODEL
# Input -> CNN -> TCN -> Dense -> Output
# ------------------------------------------------------------

inputs = Input(
    shape=X_train.shape[1:],
    name="Input"
)

# CNN
x = Conv1D(
    filters=64,
    kernel_size=3,
    padding="same",
    name="CNN_64"
)(inputs)

x = BatchNormalization()(x)
x = Activation("relu")(x)
x = Dropout(0.2)(x)

# TCN
x = tcn_block(
    x,
    filters=64,
    kernel_size=3,
    dilation_rate=2,
    dropout=0.2
)

# Pooling
x = GlobalAveragePooling1D()(x)

# Dense
x = Dense(
    64,
    activation="relu",
    name="Dense_64"
)(x)

x = Dropout(0.2)(x)

# Output
outputs = Dense(
    6,
    activation="softmax",
    name="Output"
)(x)

cnn_tcn_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="CNN_TCN_Hybrid"
)

# ------------------------------------------------------------
# 2. COMPILE
# ------------------------------------------------------------

cnn_tcn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_tcn_model.summary()

# ------------------------------------------------------------
# 3. CALLBACKS
# ------------------------------------------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=1,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

# ------------------------------------------------------------
# 4. TRAIN
# ------------------------------------------------------------

start_time = time.time()

history_cnn_tcn = cnn_tcn_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=15,
    batch_size=256,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    class_weight=None,
    shuffle=True,
    verbose=1
)

training_time = time.time() - start_time

# ------------------------------------------------------------
# 5. TEST
# ------------------------------------------------------------

test_loss, test_accuracy = cnn_tcn_model.evaluate(
    X_test,
    y_test,
    batch_size=256,
    verbose=1
)

# ------------------------------------------------------------
# 6. PREDICTIONS
# ------------------------------------------------------------

y_prob = cnn_tcn_model.predict(
    X_test,
    batch_size=256,
    verbose=1
)

y_pred = np.argmax(
    y_prob,
    axis=1
)

# ------------------------------------------------------------
# 7. METRICS
# ------------------------------------------------------------

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

print("\n" + "=" * 65)
print("CNN + TCN HYBRID - FINAL TEST RESULT")
print("=" * 65)

print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Weighted F1   : {weighted_f1:.4f}")
print(f"Training Time : {training_time / 60:.2f} minutes")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# LSTM + TCN HYBRID
# 1 LSTM + 1 TCN Block
# Build + Train + Evaluate
# ============================================================

import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Conv1D,
    BatchNormalization,
    Activation,
    Dropout,
    Add,
    GlobalAveragePooling1D,
    Dense
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    classification_report,
    f1_score
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42
tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# TCN BLOCK
# ------------------------------------------------------------

def tcn_block(x, filters=64, kernel_size=3, dilation_rate=2, dropout=0.2):

    residual = x

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        name="TCN_Conv1"
    )(x)

    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(dropout)(x)

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        name="TCN_Conv2"
    )(x)

    x = BatchNormalization()(x)

    if residual.shape[-1] != filters:
        residual = Conv1D(
            filters,
            kernel_size=1,
            padding="same"
        )(residual)

    x = Add()([x, residual])
    x = Activation("relu")(x)

    return x


# ------------------------------------------------------------
# 1. BUILD MODEL
# Input -> LSTM -> TCN -> Dense -> Output
# ------------------------------------------------------------

inputs = Input(
    shape=X_train.shape[1:],
    name="Input"
)

# LSTM
# return_sequences=True because TCN needs a sequence
x = LSTM(
    64,
    return_sequences=True,
    name="LSTM_64"
)(inputs)

x = Dropout(0.2)(x)

# TCN
x = tcn_block(
    x,
    filters=64,
    kernel_size=3,
    dilation_rate=2,
    dropout=0.2
)

# Pooling
x = GlobalAveragePooling1D()(x)

# Dense
x = Dense(
    64,
    activation="relu",
    name="Dense_64"
)(x)

x = Dropout(0.2)(x)

# Output
outputs = Dense(
    6,
    activation="softmax",
    name="Output"
)(x)

lstm_tcn_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="LSTM_TCN_Hybrid"
)

# ------------------------------------------------------------
# 2. COMPILE
# ------------------------------------------------------------

lstm_tcn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

lstm_tcn_model.summary()

# ------------------------------------------------------------
# 3. CALLBACKS
# ------------------------------------------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=1,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

# ------------------------------------------------------------
# 4. TRAIN
# ------------------------------------------------------------

start_time = time.time()

history_lstm_tcn = lstm_tcn_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=15,
    batch_size=256,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    class_weight=None,
    shuffle=True,
    verbose=1
)

training_time = time.time() - start_time

# ------------------------------------------------------------
# 5. TEST
# ------------------------------------------------------------

test_loss, test_accuracy = lstm_tcn_model.evaluate(
    X_test,
    y_test,
    batch_size=256,
    verbose=1
)

# ------------------------------------------------------------
# 6. PREDICTIONS
# ------------------------------------------------------------

y_prob = lstm_tcn_model.predict(
    X_test,
    batch_size=256,
    verbose=1
)

y_pred = np.argmax(
    y_prob,
    axis=1
)

# ------------------------------------------------------------
# 7. METRICS
# ------------------------------------------------------------

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

print("\n" + "=" * 65)
print("LSTM + TCN HYBRID - FINAL TEST RESULT")
print("=" * 65)

print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Weighted F1   : {weighted_f1:.4f}")
print(f"Training Time : {training_time / 60:.2f} minutes")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# CNN + LSTM + TCN HYBRID
# 1 CNN + 1 LSTM + 1 TCN Block
# Build + Train + Evaluate
# ============================================================

import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    LSTM,
    Dropout,
    Add,
    GlobalAveragePooling1D,
    Dense
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    classification_report,
    f1_score
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42
tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# TCN BLOCK
# ------------------------------------------------------------

def tcn_block(x, filters=64, kernel_size=3,
              dilation_rate=2, dropout=0.2):

    residual = x

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        name="TCN_Conv1"
    )(x)

    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(dropout)(x)

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        name="TCN_Conv2"
    )(x)

    x = BatchNormalization()(x)

    if residual.shape[-1] != filters:
        residual = Conv1D(
            filters,
            kernel_size=1,
            padding="same"
        )(residual)

    x = Add()([x, residual])
    x = Activation("relu")(x)

    return x


# ------------------------------------------------------------
# 1. BUILD MODEL
# Input -> CNN -> LSTM -> TCN -> Dense -> Output
# ------------------------------------------------------------

inputs = Input(
    shape=X_train.shape[1:],
    name="Input"
)

# CNN
x = Conv1D(
    filters=64,
    kernel_size=3,
    padding="same",
    name="CNN_64"
)(inputs)

x = BatchNormalization()(x)
x = Activation("relu")(x)
x = Dropout(0.2)(x)

# LSTM
# return_sequences=True because TCN needs sequence input
x = LSTM(
    64,
    return_sequences=True,
    name="LSTM_64"
)(x)

x = Dropout(0.2)(x)

# TCN
x = tcn_block(
    x,
    filters=64,
    kernel_size=3,
    dilation_rate=2,
    dropout=0.2
)

# Pooling
x = GlobalAveragePooling1D()(x)

# Dense
x = Dense(
    64,
    activation="relu",
    name="Dense_64"
)(x)

x = Dropout(0.2)(x)

# Output
outputs = Dense(
    6,
    activation="softmax",
    name="Output"
)(x)

triple_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="CNN_LSTM_TCN_Hybrid"
)

# ------------------------------------------------------------
# 2. COMPILE
# ------------------------------------------------------------

triple_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

triple_model.summary()

# ------------------------------------------------------------
# 3. CALLBACKS
# ------------------------------------------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=1,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

# ------------------------------------------------------------
# 4. TRAIN
# ------------------------------------------------------------

start_time = time.time()

history_triple = triple_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=15,
    batch_size=256,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    class_weight=None,
    shuffle=True,
    verbose=1
)

training_time = time.time() - start_time

# ------------------------------------------------------------
# 5. TEST
# ------------------------------------------------------------

test_loss, test_accuracy = triple_model.evaluate(
    X_test,
    y_test,
    batch_size=256,
    verbose=1
)

# ------------------------------------------------------------
# 6. PREDICTIONS
# ------------------------------------------------------------

y_prob = triple_model.predict(
    X_test,
    batch_size=256,
    verbose=1
)

y_pred = np.argmax(
    y_prob,
    axis=1
)

# ------------------------------------------------------------
# 7. METRICS
# ------------------------------------------------------------

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

print("\n" + "=" * 65)
print("CNN + LSTM + TCN HYBRID - FINAL TEST RESULT")
print("=" * 65)

print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Weighted F1   : {weighted_f1:.4f}")
print(f"Training Time : {training_time / 60:.2f} minutes")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# PARALLEL CNN + LSTM + TCN
# Build + Train + Evaluate
# ============================================================

import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, Activation,
    LSTM, Dropout, Add, GlobalAveragePooling1D,
    Dense, Concatenate
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    classification_report,
    f1_score
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42
tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# TCN BLOCK
# ------------------------------------------------------------

def tcn_block(x, filters=64, kernel_size=3,
              dilation_rate=2, dropout=0.2):

    residual = x

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate
    )(x)

    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(dropout)(x)

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate
    )(x)

    x = BatchNormalization()(x)

    if residual.shape[-1] != filters:
        residual = Conv1D(
            filters,
            kernel_size=1,
            padding="same"
        )(residual)

    x = Add()([x, residual])
    x = Activation("relu")(x)

    return x


# ------------------------------------------------------------
# INPUT
# ------------------------------------------------------------

inputs = Input(
    shape=X_train.shape[1:],
    name="Input"
)

# ============================================================
# BRANCH 1: CNN
# ============================================================

cnn = Conv1D(
    64,
    kernel_size=3,
    padding="same",
    name="CNN_64"
)(inputs)

cnn = BatchNormalization()(cnn)
cnn = Activation("relu")(cnn)
cnn = Dropout(0.2)(cnn)

cnn = GlobalAveragePooling1D(
    name="CNN_Pooling"
)(cnn)


# ============================================================
# BRANCH 2: LSTM
# ============================================================

lstm = LSTM(
    64,
    return_sequences=False,
    name="LSTM_64"
)(inputs)

lstm = Dropout(0.2)(lstm)


# ============================================================
# BRANCH 3: TCN
# ============================================================

tcn = tcn_block(
    inputs,
    filters=64,
    kernel_size=3,
    dilation_rate=2,
    dropout=0.2
)

tcn = GlobalAveragePooling1D(
    name="TCN_Pooling"
)(tcn)


# ============================================================
# FUSION
# ============================================================

x = Concatenate(
    name="Feature_Fusion"
)([
    cnn,
    lstm,
    tcn
])

x = Dense(
    64,
    activation="relu",
    name="Dense_64"
)(x)

x = Dropout(0.2)(x)

outputs = Dense(
    6,
    activation="softmax",
    name="Output"
)(x)

parallel_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="Parallel_CNN_LSTM_TCN"
)

# ------------------------------------------------------------
# COMPILE
# ------------------------------------------------------------

parallel_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

parallel_model.summary()

# ------------------------------------------------------------
# CALLBACKS
# ------------------------------------------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=1,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

start_time = time.time()

history_parallel = parallel_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=15,
    batch_size=256,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    class_weight=None,
    shuffle=True,
    verbose=1
)

training_time = time.time() - start_time

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

test_loss, test_accuracy = parallel_model.evaluate(
    X_test,
    y_test,
    batch_size=256,
    verbose=1
)

# ------------------------------------------------------------
# PREDICTIONS
# ------------------------------------------------------------

y_prob = parallel_model.predict(
    X_test,
    batch_size=256,
    verbose=1
)

y_pred = np.argmax(
    y_prob,
    axis=1
)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

print("\n" + "=" * 70)
print("PARALLEL CNN + LSTM + TCN - FINAL TEST RESULT")
print("=" * 70)

print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Weighted F1   : {weighted_f1:.4f}")
print(f"Training Time : {training_time / 60:.2f} minutes")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ==========================================
# MobileNet-1D Model
# ==========================================

import tensorflow as tf
from tensorflow.keras import layers, Model

tf.keras.utils.set_random_seed(42)

def build_mobilenet1d(input_shape, num_classes=6):

    inputs = layers.Input(shape=input_shape)

    # Initial convolution
    x = layers.Conv1D(
        64,
        kernel_size=3,
        padding="same",
        activation="relu"
    )(inputs)

    x = layers.BatchNormalization()(x)

    # MobileNet-style blocks
    for filters in [64, 128, 128]:

        residual = x

        x = layers.SeparableConv1D(
            filters,
            kernel_size=3,
            padding="same",
            activation="relu"
        )(x)

        x = layers.BatchNormalization()(x)

        x = layers.SeparableConv1D(
            filters,
            kernel_size=3,
            padding="same",
            activation="relu"
        )(x)

        x = layers.BatchNormalization()(x)

        # Match channels for residual connection
        if residual.shape[-1] != filters:
            residual = layers.Conv1D(
                filters,
                kernel_size=1,
                padding="same"
            )(residual)

        x = layers.Add()([x, residual])

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(
        64,
        activation="relu"
    )(x)

    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(
        num_classes,
        activation="softmax"
    )(x)

    model = Model(
        inputs,
        outputs,
        name="MobileNet1D"
    )

    return model


mobilenet_model = build_mobilenet1d(
    input_shape=X_train.shape[1:],
    num_classes=6
)

mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

mobilenet_model.summary()

In [ ]:
# ==========================================
# Train MobileNet-1D
# ==========================================

import time

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=1,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6
    )
]

start_time = time.time()

history_mobile = mobilenet_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=256,
    shuffle=True,
    callbacks=callbacks,
    verbose=1
)

training_time = (time.time() - start_time) / 60

print(
    f"\nTraining time: {training_time:.2f} minutes"
)

In [ ]:
# ==========================================
# Evaluate MobileNet-1D
# ==========================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

y_prob_mobile = mobilenet_model.predict(
    X_test,
    batch_size=256,
    verbose=1
)

y_pred_mobile = np.argmax(
    y_prob_mobile,
    axis=1
)

accuracy = accuracy_score(
    y_test,
    y_pred_mobile
)

macro_f1 = f1_score(
    y_test,
    y_pred_mobile,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_test,
    y_pred_mobile,
    average="weighted",
    zero_division=0
)

# Rare classes
f1_each = f1_score(
    y_test,
    y_pred_mobile,
    labels=np.arange(6),
    average=None,
    zero_division=0
)

mean_rare_f1 = np.mean([
    f1_each[4],
    f1_each[5]
])

print("=" * 60)
print("MOBILENET-1D RESULTS")
print("=" * 60)

print(f"Accuracy          : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Macro F1          : {macro_f1:.4f}")
print(f"Weighted F1       : {weighted_f1:.4f}")
print(f"Very Unhealthy F1 : {f1_each[4]:.4f}")
print(f"Hazardous F1      : {f1_each[5]:.4f}")
print(f"Mean Rare F1      : {mean_rare_f1:.4f}")
print(f"Training Time     : {training_time:.2f} min")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred_mobile,
        labels=np.arange(6),
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_mobile,
        labels=np.arange(6)
    )
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

models = {
    "LSTM": "lstm_model",
    "1D-CNN": "cnn_model",
    "TCN": "tcn_model",
    "CNN+LSTM": "cnn_lstm_model",
    "CNN+TCN": "cnn_tcn_model",
    "LSTM+TCN": "lstm_tcn_model",
    "CNN+LSTM+TCN": "triple_model",
    "Parallel": "parallel_model",
}

print(f"{'Model':16s} {'Acc':>8s} {'MacroF1':>9s} {'VU_F1':>8s} {'Haz_F1':>8s}")
print("-" * 54)

for name, var in models.items():
    if var not in globals():
        print(f"{name:16s}  (not found in memory)")
        continue
    m = globals()[var]
    p = np.argmax(m.predict(X_val, batch_size=256, verbose=0), axis=1)
    f1s = f1_score(y_val, p, labels=np.arange(6), average=None, zero_division=0)
    print(f"{name:16s} {accuracy_score(y_val, p)*100:7.2f}% "
          f"{f1_score(y_val, p, average='macro', zero_division=0):9.4f} "
          f"{f1s[4]:8.4f} {f1s[5]:8.4f}")

In [ ]:
p = np.argmax(mobilenet_model.predict(X_val, batch_size=256, verbose=0), axis=1)
f1s = f1_score(y_val, p, labels=np.arange(6), average=None, zero_division=0)
print(f"MobileNet-1D     {accuracy_score(y_val, p)*100:7.2f}% "
      f"{f1_score(y_val, p, average='macro', zero_division=0):9.4f} "
      f"{f1s[4]:8.4f} {f1s[5]:8.4f}")

In [ ]:
import numpy as np, tensorflow as tf
from sklearn.metrics import f1_score, accuracy_score

m = tf.keras.models.load_model('/content/drive/MyDrive/IAQ_Project_Saved/proposed_model.keras')
p = np.argmax(m.predict(X_val, batch_size=256, verbose=0), axis=1)
print(f"Val acc {accuracy_score(y_val,p)*100:.2f}%  macro-F1 {f1_score(y_val,p,average='macro',zero_division=0):.4f}")

In [ ]:
import numpy as np
y_prob_saved = np.load('/content/drive/MyDrive/IAQ_Project_Saved/y_prob_proposed.npy')
X_test_saved = np.load('/content/drive/MyDrive/IAQ_Project_Saved/X_test.npy')
y_prob_now = m.predict(X_test_saved, batch_size=256, verbose=0)
print("shapes:", y_prob_saved.shape, y_prob_now.shape)
print("max abs diff:", np.abs(y_prob_saved - y_prob_now).max())

In [ ]:
# ============================================================
# EXTERNAL DATA PREP: Bangladesh classroom (device 2)
# Exactly the same pipeline steps as the main dataset:
# Resample 3min -> 8h moving averages -> IAQI-S labels -> windows(10)
# + the saved South African scaler (no re-fit)
# No training or testing here, only preparation and class distribution
# ============================================================
import os, glob, pickle
import numpy as np, pandas as pd

DRIVE_DIR = "/content/drive/MyDrive/IAQ_Project_Saved"
WINDOW, RESAMPLE_RULE, MAX_GAP_MIN, MAX_MISSING_BINS = 10, "3min", 15, 4
FEATURES = ['CO2', 'PM2.5', 'PM10', 'T', 'RH']
LABELS = ["Good", "Moderate", "Caution", "Unhealthy", "Very Unhealthy", "Hazardous"]

# ---------- 1. Load the file ----------
hits = glob.glob(f"{DRIVE_DIR}/device2_air_quality*.csv") or \
       glob.glob("/content/drive/MyDrive/**/device2_air_quality*.csv", recursive=True)
assert hits, "File device2_air_quality*.csv not found on the drive"
CSV = hits[0]; print("CSV:", CSV)
raw = pd.read_csv(CSV)
raw["Timestamp"] = pd.to_datetime(raw["Timestamp"], errors="coerce")
raw = raw.rename(columns={"air_temperature": "T", "humidity": "RH", "pm10": "PM10", "pm2_5": "PM2.5"})
raw = raw.dropna(subset=["Timestamp"] + FEATURES).sort_values("Timestamp").drop_duplicates("Timestamp").reset_index(drop=True)
print("rows:", len(raw), "|", raw.Timestamp.min(), "->", raw.Timestamp.max())

# ---------- 2. Segments (gap > 15 minutes = new segment) ----------
gap = raw["Timestamp"].diff().dt.total_seconds().div(60)
raw["segment_id"] = (gap.isna() | (gap > MAX_GAP_MIN)).cumsum()
print("segments:", raw.segment_id.nunique())

# ---------- 3. Resample to 3 minutes within each segment + fill short gaps ----------
parts = []
for seg, g in raw.groupby("segment_id", sort=False):
    if len(g) < 2: continue
    g = g.set_index("Timestamp")[FEATURES].resample(RESAMPLE_RULE).mean()
    for col in FEATURES:
        s = g[col].copy()
        interp = s.interpolate(method="linear", limit_area="inside")
        is_nan = s.isna()
        run_id = (is_nan != is_nan.shift()).cumsum()
        run_sizes = is_nan.groupby(run_id).transform("sum")
        short = is_nan & (run_sizes <= MAX_MISSING_BINS)
        s.loc[short] = interp.loc[short]
        g[col] = s
    g = g.dropna(subset=FEATURES).rename_axis("Timestamp").reset_index()
    if len(g): parts.append(g)
data = pd.concat(parts, ignore_index=True).sort_values("Timestamp").reset_index(drop=True)
gap2 = data["Timestamp"].diff().dt.total_seconds().div(60)
data["segment_id"] = (gap2.isna() | (gap2 > MAX_GAP_MIN)).cumsum().astype("int32")
print("after resample:", len(data), "rows,", data.segment_id.nunique(), "segments")

# ---------- 4. 8-hour averages (IAQI-S predicted) ----------
out = []
for _, g in data.groupby("segment_id", sort=False):
    g = g.sort_values("Timestamp").set_index("Timestamp")
    for col in ["CO2", "PM2.5", "PM10"]:
        avg4 = g[col].rolling("4h", min_periods=2).mean()
        avg1 = g[col].rolling("1h", min_periods=2).mean()
        g[f"{col}_8h"] = (4 * avg4 + 4 * avg1) / 8
    out.append(g.reset_index())
data = pd.concat(out, ignore_index=True).dropna(subset=["CO2_8h", "PM2.5_8h", "PM10_8h"]).reset_index(drop=True)

# ---------- 5. IAQI-S (same breakpoints and EP terms as the main pipeline) ----------
def iaqi(x, pts):
    return np.interp(np.asarray(x, np.float32), np.asarray(pts, np.float32),
                     np.array([0, 50, 100, 200, 300, 400, 500], np.float32)).astype(np.float32)
PM10_P = [0, 20, 75, 150, 220, 300, 380]
PM25_P = [0, 10, 35, 70, 100, 140, 180]
CO2_P  = [300, 700, 1000, 2000, 3000, 4000, 5000]
idx = np.stack([iaqi(data["PM10_8h"], PM10_P), iaqi(data["PM2.5_8h"], PM25_P),
                iaqi(data["CO2_8h"], CO2_P)], axis=1)
base = idx.max(1); n_c = (idx > 100).sum(1)
ep1 = np.zeros(len(data), np.float32); ep1[n_c == 2] = 50; ep1[n_c == 3] = 75
ep2 = (((idx > 100) & (idx <= 200)).sum(1) * 5 + ((idx > 200) & (idx <= 300)).sum(1) * 10 +
       ((idx > 300) & (idx <= 400)).sum(1) * 20 + (idx > 400).sum(1) * 40)
data["IAQI_Composite"] = np.clip(base + ep1 + ep2, 0, 500)
cid = np.digitize(data["IAQI_Composite"].to_numpy(), bins=[50, 100, 200, 300, 400], right=True)
data["y"] = cid

# ---------- 6. South African scaler (no fit) ----------
try:
    import joblib; scaler = joblib.load(f"{DRIVE_DIR}/scaler.pkl")
except Exception:
    scaler = pickle.load(open(f"{DRIVE_DIR}/scaler.pkl", "rb"))
Xs = data[FEATURES].copy()
Xs[FEATURES] = scaler.transform(Xs[FEATURES])

# ---------- 7. 10-reading windows within the same segment ----------
Xw, yw = [], []
for _, g in data.groupby("segment_id", sort=False):
    pos = g.index.to_numpy()
    if len(pos) < WINDOW: continue
    X = Xs.loc[pos, FEATURES].to_numpy(np.float32)
    Xw.append(np.lib.stride_tricks.sliding_window_view(X, WINDOW, axis=0).transpose(0, 2, 1))
    yw.append(data.loc[pos, "y"].to_numpy()[WINDOW - 1:])
X_ext = np.concatenate(Xw); y_ext = np.concatenate(yw)

# ---------- 8. Report ----------
print("\nX_ext:", X_ext.shape, "| y_ext:", y_ext.shape)
print("\nRaw values (before scaling) for comparison with South Africa:")
print(data[FEATURES].describe().round(1).T[["mean", "min", "50%", "max"]])
print("\nClass distribution (windows):")
for i, n in enumerate(LABELS):
    c = int((y_ext == i).sum()); print(f"  {n:15s}: {c:7,}  ({100*c/len(y_ext):5.1f}%)")
np.save(f"{DRIVE_DIR}/X_ext_bd.npy", X_ext); np.save(f"{DRIVE_DIR}/y_ext_bd.npy", y_ext)
print("\nSaved X_ext_bd.npy / y_ext_bd.npy in", DRIVE_DIR)

In [ ]:
# ============================================================
# EXTERNAL VALIDATION (predict-only): Bangladesh classroom
# Loads the saved proposed_model.keras and predicts on X_ext_bd.npy
# No training, no fine-tuning, the model is not touched
# ============================================================
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)

D = "/content/drive/MyDrive/IAQ_Project_Saved"
NAMES = ["Good", "Moderate", "Caution", "Unhealthy", "Very Unhealthy", "Hazardous"]

model = tf.keras.models.load_model(f"{D}/proposed_model.keras", compile=False)
X = np.load(f"{D}/X_ext_bd.npy"); y = np.load(f"{D}/y_ext_bd.npy")
print("model input:", model.input_shape, "| X_ext:", X.shape)
assert X.shape[1:] == tuple(model.input_shape[1:]), "Input shape does not match the model"

prob = model.predict(X, batch_size=512, verbose=1)
pred = prob.argmax(1)
np.save(f"{D}/y_prob_ext_bd.npy", prob.astype(np.float32))

L = np.arange(6)
print("\n===== EXTERNAL (Bangladesh classroom) =====")
print(f"Accuracy      : {accuracy_score(y, pred)*100:.2f}%")
print(f"Macro-F1      : {f1_score(y, pred, average='macro', labels=L, zero_division=0):.4f}")
print(f"Weighted-F1   : {f1_score(y, pred, average='weighted', zero_division=0):.4f}")
print(f"Macro-Prec    : {precision_score(y, pred, average='macro', labels=L, zero_division=0):.4f}")
print(f"Macro-Recall  : {recall_score(y, pred, average='macro', labels=L, zero_division=0):.4f}")
print("\n", classification_report(y, pred, labels=L, target_names=NAMES, digits=4, zero_division=0))

cm = confusion_matrix(y, pred, labels=L)
print("Confusion matrix (rows = true, columns = predicted):")
print(pd.DataFrame(cm, index=NAMES, columns=NAMES))
print("\nRow-normalized (each row = 100%):")
print((pd.DataFrame(cm, index=NAMES, columns=NAMES).div(cm.sum(1), axis=0) * 100).round(1))
print("\nModel prediction distribution:", dict(zip(NAMES, np.bincount(pred, minlength=6).tolist())))

pd.DataFrame(cm, index=NAMES, columns=NAMES).to_csv(f"{D}/confusion_ext_bd.csv")
print("\nSaved: y_prob_ext_bd.npy, confusion_ext_bd.csv in", D)

In [ ]:
# ============================================================
# STEP A: Temporal split of the Bangladesh data + baseline (original model, no training)
# No training here
# ============================================================
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

D = "/content/drive/MyDrive/IAQ_Project_Saved"
NAMES = ["Good", "Moderate", "Caution", "Unhealthy", "Very Unhealthy", "Hazardous"]
GAP = 160                      # 8 hours = 160 windows (3 minutes each), so nothing leaks between the parts

X = np.load(f"{D}/X_ext_bd.npy"); y = np.load(f"{D}/y_ext_bd.npy")   # sorted chronologically
n = len(X)
a, b = int(n * 0.60), int(n * 0.70)
tr = slice(0, a - GAP)
va = slice(a, b - GAP)
te = slice(b, n)
X_ft, y_ft = X[tr], y[tr]; X_va, y_va = X[va], y[va]; X_te, y_te = X[te], y[te]

print(f"Fine-tune: {len(X_ft):,} | Val: {len(X_va):,} | Test: {len(X_te):,}  (gaps of {GAP} windows between them)")
dist = pd.DataFrame({"FineTune": np.bincount(y_ft, minlength=6),
                     "Val": np.bincount(y_va, minlength=6),
                     "Test": np.bincount(y_te, minlength=6)}, index=NAMES)
print("\nClass distribution:"); print(dist)

model = tf.keras.models.load_model(f"{D}/proposed_model.keras", compile=False)
pred = model.predict(X_te, batch_size=512, verbose=0).argmax(1)
L = np.arange(6)
print("\n===== Original model (no training) on the test part =====")
print(f"Accuracy : {accuracy_score(y_te, pred)*100:.2f}%")
print(f"Macro-F1 : {f1_score(y_te, pred, average='macro', labels=np.unique(y_te), zero_division=0):.4f}  (on the classes present in the test part)")
print(classification_report(y_te, pred, labels=L, target_names=NAMES, digits=4, zero_division=0))
np.savez(f"{D}/split_ext_bd.npz", a=a, b=b, gap=GAP)

In [ ]:
# ============================================================
# STEP A2: Alternating-block split of the Bangladesh data (no training here)
# Block = 960 windows (~2 days). From each block the first 160 windows (8 hours) are removed as a safety gap.
# Block assignment: 3 of every 5 -> Fine-tune, 1 -> Val, 1 -> Test
# (+ block 73, which contains Good, is kept in Fine-tune so that Good appears in training and testing)
# ============================================================
import numpy as np, pandas as pd

D = "/content/drive/MyDrive/IAQ_Project_Saved"
NAMES = ["Good", "Moderate", "Caution", "Unhealthy", "Very Unhealthy", "Hazardous"]
B, GAP = 960, 160

y = np.load(f"{D}/y_ext_bd.npy"); n = len(y)
blk = np.arange(n) // B
role = np.where(blk % 5 == 4, "te", np.where(blk % 5 == 3, "va", "ft"))
good_blocks = sorted(set(blk[y == 0]))
print("Blocks containing Good:", good_blocks)
role[blk == good_blocks[0]] = "ft"          # first block containing Good -> training
keep = (np.arange(n) % B) >= GAP

idx = {r: np.where((role == r) & keep)[0] for r in ["ft", "va", "te"]}
for r, v in idx.items():
    print(f"{r}: {len(v):,} windows")

dist = pd.DataFrame({"FineTune": np.bincount(y[idx['ft']], minlength=6),
                     "Val": np.bincount(y[idx['va']], minlength=6),
                     "Test": np.bincount(y[idx['te']], minlength=6)}, index=NAMES)
print("\nClass distribution:"); print(dist)
assert not (set(idx['ft']) & set(idx['te'])) and not (set(idx['ft']) & set(idx['va'])) and not (set(idx['va']) & set(idx['te']))

np.savez(f"{D}/block_split_ext_bd.npz", ft=idx['ft'], va=idx['va'], te=idx['te'])
print("\nSaved block_split_ext_bd.npz in", D)

In [ ]:
# ============================================================
# DOUBLE TRAINING (fine-tuning) on Bangladesh + evaluation + forgetting check
# The original proposed_model.keras is not touched: the output is saved to a new file
# ============================================================
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42; tf.keras.utils.set_random_seed(SEED)
D = "/content/drive/MyDrive/IAQ_Project_Saved"
NAMES = ["Good", "Moderate", "Caution", "Unhealthy", "Very Unhealthy", "Hazardous"]
L = np.arange(6)

X = np.load(f"{D}/X_ext_bd.npy"); y = np.load(f"{D}/y_ext_bd.npy")
sp = np.load(f"{D}/block_split_ext_bd.npz")
Xf, yf, Xv, yv, Xt, yt = X[sp['ft']], y[sp['ft']], X[sp['va']], y[sp['va']], X[sp['te']], y[sp['te']]
Xsa, ysa = np.load(f"{D}/X_test.npy"), np.load(f"{D}/y_test.npy")      # South African Test set (for the forgetting check)
print(f"Fine-tune {len(Xf):,} | Val {len(Xv):,} | BD-Test {len(Xt):,} | SA-Test {len(Xsa):,}")

def metrics(m, Xe, ye):
    p = m.predict(Xe, batch_size=512, verbose=0).argmax(1)
    return p, accuracy_score(ye, p) * 100, f1_score(ye, p, average="macro", labels=L, zero_division=0)

orig = tf.keras.models.load_model(f"{D}/proposed_model.keras", compile=False)
ft   = tf.keras.models.load_model(f"{D}/proposed_model.keras", compile=False)   # second copy for training
ft.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

h = ft.fit(Xf, yf, validation_data=(Xv, yv), epochs=10, batch_size=256, shuffle=True, verbose=2,
           callbacks=[EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)])
ft.save(f"{D}/proposed_model_ft_bd.keras")

p0, a0, f0 = metrics(orig, Xt, yt)
p1, a1, f1 = metrics(ft,   Xt, yt)
_, sa0, sf0 = metrics(orig, Xsa, ysa)
_, sa1, sf1 = metrics(ft,   Xsa, ysa)

print("\n===== Bangladesh - test part (14,400 windows) =====")
print(f"Original model      : Acc {a0:.2f}% | Macro-F1 {f0:.4f}")
print(f"After fine-tuning   : Acc {a1:.2f}% | Macro-F1 {f1:.4f}")
print("\nF1 per class (original -> after training):")
f1o = f1_score(yt, p0, average=None, labels=L, zero_division=0)
f1n = f1_score(yt, p1, average=None, labels=L, zero_division=0)
for i, n in enumerate(NAMES): print(f"  {n:15s}: {f1o[i]:.4f} -> {f1n[i]:.4f}")
print("\n", classification_report(yt, p1, labels=L, target_names=NAMES, digits=4, zero_division=0))
print("Confusion matrix (after training):")
print(pd.DataFrame(confusion_matrix(yt, p1, labels=L), index=NAMES, columns=NAMES))

print("\n===== Forgetting check: South African Test =====")
print(f"Original model      : Acc {sa0:.2f}% | Macro-F1 {sf0:.4f}")
print(f"After fine-tuning   : Acc {sa1:.2f}% | Macro-F1 {sf1:.4f}")
print("\nSaved: proposed_model_ft_bd.keras (the original is unchanged)")

In [ ]:
# ============================================================
# Curves of the main model (South Africa) — no training
# 1) ROC per class (One-vs-Rest) + AUC   2) Precision-Recall per class + AP
# 3) Training curves (Loss/Accuracy) if history exists in the session
# Reads from the drive: y_test.npy and y_prob_proposed.npy (saved probabilities of the final model)
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize

D = "/content/drive/MyDrive/IAQ_Project_Saved"
NAMES = ["Good", "Moderate", "Caution", "Unhealthy", "Very Unhealthy", "Hazardous"]
COLS  = ["#1b9e77", "#7570b3", "#e6ab02", "#d95f02", "#e7298a", "#a50f15"]

y = np.load(f"{D}/y_test.npy"); p = np.load(f"{D}/y_prob_proposed.npy")
assert len(y) == len(p), f"Lengths differ: {len(y)} vs {len(p)}"
Y = label_binarize(y, classes=np.arange(6))
plt.rcParams.update({"font.size": 10})

# ---------- 1) ROC ----------
fig, ax = plt.subplots(figsize=(6.4, 5.4)); aucs = []
for k in range(6):
    fpr, tpr, _ = roc_curve(Y[:, k], p[:, k]); a = auc(fpr, tpr); aucs.append(a)
    ax.plot(fpr, tpr, color=COLS[k], lw=1.8, label=f"{NAMES[k]} (AUC = {a:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.9)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title(f"ROC curves (one-vs-rest), macro AUC = {np.mean(aucs):.3f}")
ax.legend(loc="lower right", frameon=False, fontsize=9); ax.grid(alpha=.25)
fig.tight_layout(); fig.savefig(f"{D}/fig_roc.png", dpi=300); plt.show()

# ---------- 2) Precision-Recall ----------
fig, ax = plt.subplots(figsize=(6.4, 5.4)); aps = []
for k in range(6):
    pr, rc, _ = precision_recall_curve(Y[:, k], p[:, k]); ap = average_precision_score(Y[:, k], p[:, k]); aps.append(ap)
    ax.plot(rc, pr, color=COLS[k], lw=1.8, label=f"{NAMES[k]} (AP = {ap:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_ylim(0, 1.02)
ax.set_title(f"Precision-Recall curves, mean AP = {np.mean(aps):.3f}")
ax.legend(loc="lower left", frameon=False, fontsize=9); ax.grid(alpha=.25)
fig.tight_layout(); fig.savefig(f"{D}/fig_pr.png", dpi=300); plt.show()

# ---------- 3) Training curves (if history exists in memory) ----------
hist = None
for name in ["history_triple", "history_proposed", "history_cnn_lstm_tcn", "history"]:
    if name in globals() and hasattr(globals()[name], "history"):
        hist = globals()[name].history; print("Used:", name); break
if hist is None:
    print("Training curves: no history variable found in the session. (Training is not repeated; the history of the original run is needed.)")
else:
    ep = np.arange(1, len(hist["loss"]) + 1)
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    axs[0].plot(ep, hist["loss"], "o-", label="Train"); axs[0].plot(ep, hist["val_loss"], "s-", label="Validation")
    axs[0].set_xlabel("Epoch"); axs[0].set_ylabel("Loss"); axs[0].set_title("Loss")
    axs[1].plot(ep, np.array(hist["accuracy"]) * 100, "o-", label="Train"); axs[1].plot(ep, np.array(hist["val_accuracy"]) * 100, "s-", label="Validation")
    axs[1].set_xlabel("Epoch"); axs[1].set_ylabel("Accuracy (%)"); axs[1].set_title("Accuracy")
    for a in axs: a.legend(frameon=False); a.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(f"{D}/fig_training_curves.png", dpi=300); plt.show()

print("\nSaved in", D, ": fig_roc.png, fig_pr.png" + (", fig_training_curves.png" if hist is not None else ""))
print("AUC :", dict(zip(NAMES, np.round(aucs, 4)))); print("AP  :", dict(zip(NAMES, np.round(aps, 4))))

In [ ]:
# Class counts and location info for the two DALTON external sets
import numpy as np, pandas as pd

D = "/content/drive/MyDrive/IAQ_Project_Saved"
NAMES = ["Good", "Moderate", "Caution", "Unhealthy", "Very Unhealthy", "Hazardous"]

for tag in ["dalton", "dalton2"]:
    y = np.load(f"{D}/y_ext_{tag}.npy")
    node = np.load(f"{D}/node_ext_{tag}.npy", allow_pickle=True)
    print(f"\n===== {tag} =====")
    print("windows:", len(y), "| node array:", node.shape, node.dtype)
    print(pd.Series(np.bincount(y, minlength=6), index=NAMES))
    u, c = np.unique(node, return_counts=True)
    print("unique nodes/locations:", len(u))
    print(pd.Series(c, index=u).head(30))
    print(pd.read_csv(f"{D}/confusion_ext_{tag}.csv", index_col=0))

In [ ]:
# Ablation of the hybrid model: (1) no TCN block, (2) non-causal TCN block, 5 seeds each
import os, time, json
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, Activation,
                                     LSTM, Dropout, Add, GlobalAveragePooling1D, Dense)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import f1_score, accuracy_score

SEEDS = [42, 7, 123, 2024, 1]
SAVE_DIR = "/content/drive/MyDrive/IAQ_Project_Saved/multi_seed_ablation"
os.makedirs(SAVE_DIR, exist_ok=True)

# TCN residual block; padding can be causal (default) or same (non-causal)
def tcn_block(x, padding="causal", filters=64, kernel_size=3, dilation_rate=2, dropout=0.2):
    res = x
    x = Conv1D(filters, kernel_size, padding=padding, dilation_rate=dilation_rate)(x)
    x = BatchNormalization()(x); x = Activation("relu")(x); x = Dropout(dropout)(x)
    x = Conv1D(filters, kernel_size, padding=padding, dilation_rate=dilation_rate)(x)
    x = BatchNormalization()(x)
    if res.shape[-1] != filters:
        res = Conv1D(filters, 1, padding="same")(res)
    return Activation("relu")(Add()([x, res]))

# Hybrid backbone with the TCN block optional and its padding configurable
def build_variant(use_tcn=True, tcn_padding="causal"):
    i = Input(shape=X_train.shape[1:])
    x = Conv1D(64, 3, padding="same")(i)
    x = Dropout(0.2)(Activation("relu")(BatchNormalization()(x)))
    x = Dropout(0.2)(LSTM(64, return_sequences=True)(x))
    if use_tcn:
        x = tcn_block(x, padding=tcn_padding)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.2)(Dense(64, activation="relu")(x))
    return Model(i, Dense(6, activation="softmax")(x))

VARIANTS = {
    "no_TCN":        dict(use_tcn=False),
    "TCN_noncausal": dict(use_tcn=True, tcn_padding="same"),
}

csv_path = f"{SAVE_DIR}/results_ablation.csv"
done = set()
if os.path.exists(csv_path):
    _d = pd.read_csv(csv_path)
    done = set(zip(_d["variant"], _d["seed"]))

for seed in SEEDS:
    for name, kw in VARIANTS.items():
        if (name, seed) in done:
            print(f"{name} seed {seed}: already done"); continue
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)
        m = build_variant(**kw)
        m.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        cb = [EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True),
              ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6)]
        t0 = time.time()
        h = m.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15,
                  batch_size=256, class_weight=None, shuffle=True, callbacks=cb, verbose=0)
        minutes = (time.time() - t0) / 60
        pv = m.predict(X_val, batch_size=256, verbose=0)
        pt = m.predict(X_test, batch_size=256, verbose=0)
        yv, yt = np.argmax(pv, 1), np.argmax(pt, 1)
        f1c = f1_score(y_test, yt, average=None, labels=np.arange(6), zero_division=0)
        row = {"variant": name, "seed": seed,
               "val_macro_f1": f1_score(y_val, yv, average="macro"),
               "test_acc": accuracy_score(y_test, yt) * 100,
               "test_macro_f1": f1_score(y_test, yt, average="macro"),
               "epochs_run": len(h.history["loss"]), "minutes": minutes}
        for k, n in enumerate(["Good", "Moderate", "Caution", "Unhealthy", "VeryUnhealthy", "Hazardous"]):
            row["f1_" + n] = f1c[k]
        print(f"{name:14s} seed {seed:5d} | test macro-F1 {row['test_macro_f1']:.4f} | acc {row['test_acc']:.2f} | epochs {row['epochs_run']} | {minutes:.1f} min", flush=True)
        pd.DataFrame([row]).to_csv(csv_path, mode="a", header=not os.path.exists(csv_path), index=False)

df = pd.read_csv(csv_path)
print(df.groupby("variant")[["val_macro_f1", "test_macro_f1", "test_acc", "f1_Hazardous"]].agg(["mean", "std"]).round(4))